# 01 — FinQA as a retrieval benchmark: data, ambiguity, chunking, golden set

Companion to the [Block 2 course](../README.md). This notebook builds the **evaluation set** and
shows, with the real data, the three problems the protocol had to solve before measuring anything:

1. FinQA ships **pages**, not reports, and its questions were written *looking at the page*.
2. In a pooled index, most questions are **ambiguous** unless we say which company they are about.
3. **Chunking** decides what a "golden chunk" even is, so the gold mapping must be defined per strategy.

Everything comes from `ragbench` (data loading, chunking, audit sample); this notebook only displays.

In [1]:
import collections
import os
import statistics as st

os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import pandas as pd

from ragbench.audit import audited_sample, gold_type
from ragbench.b2 import load_bench
from ragbench.chunking import is_headerless, linearize_page

pd.set_option("display.max_colwidth", 120)
bench = load_bench()
pages, questions = bench.pages, bench.questions
print(f"{len(pages)} unique pages, {len(questions)} questions, "
      f"{len({p.ticker for p in pages.values()})} companies")

380 unique pages, 1147 questions, 100 companies


## 1. What a FinQA record looks like

One record = one question + the full content of the page it was written on: sentences before the
table (`pre_text`), the table, sentences after (`post_text`). Gold evidence `gold_inds` points to
sentences (`text_i`, indexed over pre_text then post_text) and table rows (`table_i`).

In [2]:
q = next(x for x in questions if x.qid == "ETR/2016/page_23.pdf-2")
page = pages[q.page_id]
print("Q:", q.question)
print("program:", q.program, "->", q.answer)
print("gold:", q.gold_keys)
print("\ntable:")
for i, row in enumerate(page.table):
    print(f"  table_{i}: {' | '.join(row)}")

Q: what is the net change in net revenue during 2015 for entergy corporation?
program: subtract(5829, 5735) -> 94.0
gold: ('table_1', 'table_8')

table:
  table_0:  | amount ( in millions )
  table_1: 2014 net revenue | $ 5735
  table_2: retail electric price | 187
  table_3: volume/weather | 95
  table_4: waterford 3 replacement steam generator provision | -32 ( 32 )
  table_5: miso deferral | -35 ( 35 )
  table_6: louisiana business combination customer credits | -107 ( 107 )
  table_7: other | -14 ( 14 )
  table_8: 2015 net revenue | $ 5829


## 2. Questions are written for a page, not for a corpus

How often does a question say *which company* it is about? We check whether the company name
(or its first word) appears in the question.

In [3]:
def names_company(q):
    name = bench.company(q.page_id).lower().replace("the ", "")
    return name.split()[0] in q.question.lower()

share = sum(map(names_company, questions)) / len(questions)
print(f"questions naming their company: {share:.1%}")
for x in questions[100:106]:
    print(f"  [{x.ticker}] {x.question}")

questions naming their company: 7.6%
  [DISCA] what was the percentage cumulative total shareholder return on disca common stock for the five year period ended december 31 , 2014?
  [C] what percent of net interest revenue where total operating expenses in 2008?
  [GPN] what percentage of lease payments will be paid out in the first year?
  [GPN] what is the percentage change in the total fair value of non-vested shares from 2009 to 2010?
  [AON] what is portion of the total consideration transferred is dedicated to goodwill?
  [EOG] considering the years 2016-2018 , what is the average ending balance for valuation allowances for deferred income tax assets?


A question like *"what is the percentage change in rental expense from 2015 to 2016?"* has a valid
answer in dozens of the 380 pages. Scoring a retriever on whether it finds *the* page the question
was written on would measure ambiguity, not retrieval. The protocol therefore prefixes the company
name, the information an analyst at Armor Gestion always has:

In [4]:
for x in questions[100:103]:
    print(f"{bench.company(x.page_id)}: {x.question}")

Discovery Communications: what was the percentage cumulative total shareholder return on disca common stock for the five year period ended december 31 , 2014?
Citigroup: what percent of net interest revenue where total operating expenses in 2008?
Global Payments: what percentage of lease payments will be paid out in the first year?


Not with the year, though: the FinQA folder year (`ETR/2016/...`) is not reliably the fiscal year
of the report. The Entergy page above is filed under 2016 but its table covers 2014–2015.
The company names themselves come from `configs/finqa_tickers.csv`, checked against the pages:
CB pages are **ACE Limited** (before the 2016 Chubb merger) and HWM pages are **Alcoa** (2015) or
**Arconic** (2017–2018).

How many companies have pages from several years, i.e. realistic near-duplicates?

In [5]:
years = collections.defaultdict(set)
for p in pages.values():
    years[p.ticker].add(p.page_id.split("/")[1])
multi = sum(len(v) > 1 for v in years.values())
print(f"{multi} of {len(years)} companies have pages from several years")

71 of 100 companies have pages from several years


## 3. Two chunkings, two definitions of the golden chunk

In [6]:
corpora = {s: bench.chunk(s) for s in ("naive", "table_aware")}
rows = []
for s, c in corpora.items():
    lens = [bench.count(ch.text) for ch in c.chunks]
    sets = [len(c.gold_sets(q.page_id, q.gold_keys)) for q in questions]
    rows.append({"strategy": s, "chunks": len(c.chunks), "mean tokens": round(st.mean(lens), 1),
                 "median tokens": st.median(lens), "gold sets / question": round(st.mean(sets), 2)})
pd.DataFrame(rows)

,strategy,chunks,mean tokens,median tokens,gold sets / question
0,naive,1702,178.4,200.0,1.22
1,table_aware,4024,80.2,43.0,1.59


Two things to notice before reading any benchmark number:

- **Naive chunks are ~2× longer.** At equal k, naive sends twice as many tokens to the LLM.
- **Naive chunks merge gold facts** (1.22 gold sets per question vs 1.59): two numbers needed for
  a subtraction often fall into the same 200-token window. That makes `pass_all@k` *easier* for
  naive at equal k. Hence amendment A2: also compare at a fixed **token budget**.

### The same page, chunked both ways

In [7]:
for s, c in corpora.items():
    print(f"===== {s}")
    for ch in [x for x in c.chunks if x.page_id == q.page_id][:6]:
        print(f"  [{ch.kind:5}] {ch.text[:150].replace(chr(10), ' ⏎ ')}")

===== naive
  [text ] entergy corporation and subsidiaries management 2019s financial discussion and analysis a result of the entergy louisiana and entergy gulf states loui
  [text ] portion of the assets associated with the waterford 3 replacement steam generator project is no longer probable of recovery . ⏎ see note 14 to the finan
  [mixed] into with the mississippi public utilities staff , subsequently approved by the mpsc , in which entergy mississippi agreed not to pursue recovery of t
  [text ] an increase in energy efficiency rider revenue primarily due to increases in the energy efficiency rider at entergy arkansas , as approved by the apsc
===== table_aware
  [text ] entergy corporation and subsidiaries management 2019s financial discussion and analysis a result of the entergy louisiana and entergy gulf states loui
  [text ] results of operations for 2015 also include the sale in december 2015 of the 583 mw rhode island state energy center for a realized gain of $ 154 mill
  

### What naive chunking does to the evidence

In [8]:
c = corpora["naive"]
whole = split = rows_n = rows_hdr = 0
for x in questions:
    pg = pages[x.page_id]
    text, spans = linearize_page(pg)
    sp = {k: (s, e) for k, s, e in spans}
    for k in x.gold_keys:
        chunk = c.chunks[min(c.gold_map[x.page_id][k])].text
        unit = text[sp[k][0]:sp[k][1]]
        whole += bool(unit.strip()) and unit in chunk
        split += not (unit.strip() and unit in chunk)
        if k.startswith("table_") and k != "table_0":
            rows_n += 1
            header = text[sp["table_0"][0]:sp["table_0"][1]]
            rows_hdr += bool(header.strip()) and header in chunk
print(f"gold facts cut across two naive chunks: {split}/{whole + split} ({split / (whole + split):.1%})")
print(f"gold table rows separated from their header row: {rows_n - rows_hdr}/{rows_n} "
      f"({(rows_n - rows_hdr) / rows_n:.1%})")

gold facts cut across two naive chunks: 188/1903 (9.9%)
gold table rows separated from their header row: 356/1305 (27.3%)


A row like `2015 net revenue | $ 5829` without its header does not say *in millions* or *which
column is which year*. That is the Armor Gestion failure in one line.

### A data quirk the audit caught: tables without a header

FinQA keeps the first table row at index 0 even when it is data. Using it as a header wrote a
fake column name in front of every value (`trade name | $ 2800: 2800`). `is_headerless` detects
these tables (value cells all numeric, years excepted):

In [9]:
headerless = [p for p in pages.values() if len(p.table) > 1 and is_headerless(p.table)]
print(f"{len(headerless)} of {sum(len(p.table) > 1 for p in pages.values())} tables are headerless")
for p in headerless[:3]:
    print("  ", p.table[0], "||", p.table[1])

38 of 379 tables are headerless
   ('value of metavante common stock', '$ 4066.4') || ('value of metavante stock awards', '121.4')
   ('balance at january 1 2013', '$ 180993') || ('increases in current period tax positions', '27229')
   ('beginning balance as of december 1 2007', '$ 201808') || ('gross increases in unrecognized tax benefits 2013 prior year tax positions', '14009')


Other FinQA text artifacts you will see in chunks: `2019s` is a mangled `’s` (unicode U+2019),
`201c` / `201d` are curly quotes, and `2014` often stands for an em-dash. They are identical
across all configurations, so they do not bias comparisons, but they do add noise to BM25.

## 4. The audited subset (25 questions)

Stratified by gold type, seed 0. Each mapping below was read by hand for both chunkings; the
audit found the headerless-table issue above (amendment A3) and no mapping error otherwise.

In [10]:
sample = audited_sample(questions)
print(collections.Counter(gold_type(x) for x in sample), "vs population",
      {k: round(v / len(questions), 2) for k, v in collections.Counter(map(gold_type, questions)).items()})
audit = []
for x in sample:
    for s, c in corpora.items():
        for gs in c.gold_sets(x.page_id, x.gold_keys):
            audit.append({"qid": x.qid, "strategy": s, "question": x.question[:70],
                          "acceptable": len(gs), "chunk": c.chunks[min(gs)].text[:110]})
pd.DataFrame(audit).head(20)

Counter({'table': 16, 'text': 6, 'table+text': 3}) vs population {'table': 0.62, 'text': 0.25, 'table+text': 0.14}


,qid,strategy,question,acceptable,chunk
0,AMAT/2018/page_33.pdf-1,naive,what is the roi for applied materials if the investment made on octobe,1,"reinvestment of dividends , if any .\ndollar amounts in the graph are rounded to the nearest whole dollar .\nthe"
1,AMAT/2018/page_33.pdf-1,table_aware,what is the roi for applied materials if the investment made on octobe,1,applied materials | 10/27/2013: 100.00 | 10/26/2014: 121.04 | 10/25/2015: 96.67 | 10/30/2016: 171.69 | 10/29/2
2,BKNG/2016/page_33.pdf-4,naive,what was the percentage change in priceline group for the five year pe,1,measurement point december 31 the priceline group nasdaq composite index s&p 500 rdg internet composite .\nmeas
3,BKNG/2016/page_33.pdf-4,table_aware,what was the percentage change in priceline group for the five year pe,1,2011 | the priceline group inc .: 100.00 | nasdaqcomposite index: 100.00 | s&p 500index: 100.00 | rdg internet
4,BKNG/2016/page_33.pdf-4,table_aware,what was the percentage change in priceline group for the five year pe,1,2016 | the priceline group inc .: 313.45 | nasdaqcomposite index: 216.54 | s&p 500index: 198.18 | rdg internet
5,BLK/2014/page_120.pdf-2,naive,what percent of future notes are due by 2017?,1,and contingencies operating lease commitments the company leases its primary office spaces under agreements th
6,BLK/2014/page_120.pdf-2,table_aware,what percent of future notes are due by 2017?,1,"interest is payable semi-annually in arrears on march 15 and september 15 of each year , or approximately $ 44"
7,BLK/2014/page_120.pdf-2,table_aware,what percent of future notes are due by 2017?,1,2015 | amount: $ 126
8,BLK/2014/page_120.pdf-2,table_aware,what percent of future notes are due by 2017?,1,2016 | amount: 111
9,BLK/2014/page_120.pdf-2,table_aware,what percent of future notes are due by 2017?,1,2017 | amount: 112
